# Domain-Adaptive GPT-2 — Phase 1: Clinical Continued Pretraining

Run on **Google Colab (T4 GPU)**. This notebook takes the public clinical text corpus (PubMed abstracts + seed references in `knowledge_base/chunks.jsonl`) performs GPT-2 continued pretraining.

The output is a checkpoint `gpt2-clinical-base` saved to Google Drive, later used for Phase 2 (RAG SFT report generation).

In [ ]:
!nvidia-smi -L
!pip -q install transformers datasets accelerate sentencepiece

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
REPO = '/content/medical-cdss'
DRIVE = '/content/drive/MyDrive/medical-cdss'
OUT = f'{DRIVE}/gpt2-clinical-base'
os.makedirs(DRIVE, exist_ok=True)
if not os.path.isdir(REPO):
    !git clone https://github.com/sharon504/medical-cdss.git $REPO
%cd $REPO

In [ ]:
import json, random
from pathlib import Path

from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorForLanguageModeling, Trainer, TrainingArguments

BASE = 'gpt2-medium'   # fall back to 'gpt2' if the free tier runs out of RAM
tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE)

# corpus: all knowledge-base passage texts (updatable without retraining)
chunks = [json.loads(l) for l in open(f'{REPO}/knowledge_base/chunks.jsonl')]
texts = [c['text'] for c in chunks]
print(f'{len(texts)} passages loaded')

random.seed(42)
random.shuffle(texts)
ds = Dataset.from_dict({'text': texts})
ds = ds.map(lambda ex: tok(ex['text'], truncation=True, max_length=512),
            batched=True, remove_columns=['text'])
print(ds)

In [ ]:
collator = DataCollatorForLanguageModeling(tok, mlm=False)

args = TrainingArguments(
    output_dir=f'{DRIVE}/ckpt-clinical-base',
    overwrite_output_dir=True,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=3e-4,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type='linear',
    fp16=True,
    logging_steps=20,
    save_strategy='epoch',
    report_to='none',
)

trainer = Trainer(model=model, args=args, train_dataset=ds, data_collator=collator)
trainer.train()
trainer.save_model(OUT)
tok.save_pretrained(OUT)
print('saved domain-adaptive base ->', OUT)

In [ ]:
# quick sanity generation
from transformers import pipeline
gen = pipeline('text-generation', model=OUT, device=0)
print(gen('A 52 year old woman presents with fatigue, menorrhagia and low MCV,',
          max_new_tokens=80, do_sample=True, temperature=0.7)[0]['generated_text'])

Next: notebook `02_rag_sft_gpt2.ipynb` fine-tunes this base on
evidence + patient summary → structured report pairs.